# fetch_url as a tool, or only as ml.fetch inside exec

Spec: `tests/e2e/bench/specs/tools-into-js.bench.ts`. Three arms, each with `answer`: `tool` (fetch_url alone), `exec`
(exec alone; its description names `ml.fetch`) and `both` (fetch_url and exec). Two tasks on the fixture site:
`sales-total` (sum a 3,000-row CSV) and `find-out` (one fact from another page).

Questions:

1. Does a run lose correctness, steps, tokens or time when fetch_url is not a tool (`exec` against `tool`)?
2. In the `exec` arm, do models find `ml.fetch` (on which turn, after reading `agent_api_docs` or not), or fall back to
   a page-world `fetch(`?
3. In the `both` arm, which reader do they reach for first, and do they switch?

Every run is classified by the readers it used, in order: `fetch_url` (sub-kinds `fetch_url+ask`, `fetch_url+pipe`),
`ml.fetch` (an exec whose `js` matches `\bml\.fetch\s*\(`), `page-fetch` (an exec calling bare `fetch(` or
`XMLHttpRequest` without `ml.fetch`), `api-docs` (`agent_api_docs`), `exec-other` (any other exec).

In [1]:
import json, re
from pathlib import Path
import pandas as pd
import wmlnb

HERE = wmlnb.ROOT / "notebooks" / "bench"
SWEEP = "tools-into-js"
EXTRACT = HERE / "data" / "tools-into-js" / "runs.json"
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

## The runs, pinned

One row per run, from each run's `cell.json` (outcome, tokens, time) and `run.json` (its steps). An earlier launch of
the same sweep ran with `navigate` on and left stale cells. A cell is **excluded** when any of these holds: it sits under
`history/` (the harness moves a replaced cell there), its run has a `navigate` step (no arm offers it, so only the broken
launch can produce one), or its session started before the first `sweeps.jsonl` record carrying the latest record's
fingerprint (it ran under an older spec). The cell itself carries no fingerprint, so the third test uses the session
time. The excluded cells are listed in `excluded`.

In [2]:
FETCH_JS = re.compile(r"\bml\.fetch\s*\(")
PAGE_JS = re.compile(r"(?<![\w.])fetch\s*\(|XMLHttpRequest")
URL_READERS = ("fetch_url", "ml.fetch", "page-fetch")
# The literal URL an ml.fetch call names: whatever quote opens it, then URL characters.
REL_JS = re.compile(r"ml\.fetch\(\s*[^\w\s(]([\w/:.?=&%~-]+)")


def classify(step):
    """The kind of one tool step: how it read a URL, or its tool name."""
    tool, args = step.get("tool"), step.get("arguments") or {}
    if tool == "fetch_url":
        return "fetch_url+ask" if args.get("ask") else "fetch_url+pipe" if args.get("pipe") else "fetch_url"
    if tool == "exec":
        js = str(args.get("js", ""))
        return "ml.fetch" if FETCH_JS.search(js) else "page-fetch" if PAGE_JS.search(js) else "exec-other"
    if tool == "agent_api_docs":
        return "api-docs"
    return tool


def base(kind):
    return "fetch_url" if kind.startswith("fetch_url") else kind


def read_runs():
    root = wmlnb.find_sweep(SWEEP)
    sweeps = [json.loads(l) for l in (root / "sweeps.jsonl").read_text().splitlines() if l.strip()]
    sweeps = [s for s in sweeps if s.get("op") == "sweep"]
    latest = sweeps[-1]["fingerprint"]
    since = min(s["at"] for s in sweeps if s["fingerprint"] == latest)
    rows, excluded = [], []
    for cell in sorted(root.rglob("cell.json")):
        rel = str(cell.relative_to(root))
        c = json.loads(cell.read_text())
        m = c.get("measurement") or {}
        run = cell.parent / "run.json"
        sess = json.loads(run.read_text())["session"] if run.exists() else {"steps": [], "createdAt": ""}
        steps = [s for s in sess["steps"] if s.get("tool")]
        why = ("history/" if rel.startswith("history/") else
               "navigate step" if any(s["tool"] == "navigate" for s in steps) else
               "older fingerprint" if sess.get("createdAt", "") < since else None)
        if why:
            excluded.append({"cell": rel, "why": why})
            continue
        kinds = [classify(s) for s in steps]
        readers = [k for k in kinds if k not in ("answer",) and (base(k) in URL_READERS or k in ("api-docs", "exec-other"))]
        used = [base(k) for k in kinds if base(k) in URL_READERS]
        mf = next((i + 1 for i, k in enumerate(kinds) if k == "ml.fetch"), None)
        execs = [s for s in steps if s["tool"] == "exec"]
        tok = m.get("tokens") or {}
        gen_ms = sum((s.get("usage") or {}).get("genMs") or 0 for s in sess["steps"])
        rows.append({
            "model": c["combo"]["model"], "arm": c["combo"]["arm"], "task": c["taskId"], "repeat": c["repeat"],
            "correct": bool(m.get("succeeded")), "steps": m.get("steps"), "run_s": (m.get("runMs") or 0) / 1000,
            "gen_s": gen_ms / 1000, "prompt_tokens": tok.get("prompt"), "completion_tokens": tok.get("completion"),
            "readers": readers, "first_reader": used[0] if used else None, "switched": len(set(used)) > 1,
            "mlfetch_turn": mf, "before_mlfetch": kinds[:mf - 1] if mf else None,
            "docs_before_mlfetch": ("api-docs" in kinds[:mf - 1]) if mf else None,
            "failed_first_exec": ("Error" in str(execs[0].get("result"))) if execs else None,
            "unknown_tool_calls": sum("no tool named" in str(s.get("result")) for s in sess["steps"]),
            # The runner's own error (a run that hit the spec's 300 s cap gives no agent-result), scored as not correct.
            "timed_out": "timed out" in str(c.get("error") or m.get("error") or ""),
            # ml.fetch called with a relative URL, and refused by the worker (sw-fetch-url.ts parses it with no base).
            "relative_mlfetch": sum(len([u for u in REL_JS.findall(str((s.get("arguments") or {}).get("js", ""))) if not u.startswith(("http://", "https://"))]) for s in steps if s["tool"] == "exec"),
            "relative_refused": sum("not a valid URL" in str(s.get("result")) for s in steps if s["tool"] == "exec"),
        })
    return {"rows": rows, "excluded": excluded, "latest_fingerprint": latest, "since": since}


data = wmlnb.extract(EXTRACT, read_runs, source={"sweep": SWEEP, "sweep_root": "tests/e2e/artifacts/bench/tools-into-js"})
runs = pd.DataFrame(data["rows"])
print(len(runs), "runs kept; excluded:", data["excluded"], "; latest fingerprint", data["latest_fingerprint"], "since", data["since"])
pd.DataFrame(wmlnb.provenance(EXTRACT))

90 runs kept; excluded: [{'cell': 'history/sales-total/model-gemma4-12b_arm-tool/r0/2026-11-08T09-30-11-344Z-02487805/cell.json', 'why': 'history/'}, {'cell': 'history/sales-total/model-gemma4-31b_arm-tool/r0/2026-11-08T09-32-07-183Z-1cb55b48/cell.json', 'why': 'history/'}] ; latest fingerprint 0bfc76bb6caf8195b9d4ac85774a54544dd9b51d since 2026-11-08T09:37:16.352Z


,extract,sha256,rows,extracted_at,commit,sweep,sweep_root
0,notebooks/bench/data/tools-into-js/runs.json,99709619424878ea,4,2026-11-08T11:44:03+00:00,0bfc76bb6caf,tools-into-js,tests/e2e/artifacts/bench/tools-into-js


## Runs per cell (n)

In [3]:
G = ["model", "task", "arm"]
runs.groupby(G).size().unstack("arm", fill_value=0)

arm                                    both  exec  tool
model                     task                         
gemma4:12b                find-out        5     5     5
                          sales-total     5     5     5
gemma4:31b                find-out        5     5     5
                          sales-total     5     5     5
qwen3.8-flash-next:vision find-out        5     5     5
                          sales-total     5     5     5

## Correct rate, model x task x arm

In [4]:
def rate(df):
    g = df.groupby(G).correct.agg(["sum", "count"])
    g["correct"] = g["sum"].astype(int).astype(str) + "/" + g["count"].astype(str)
    g["rate"] = (g["sum"] / g["count"]).round(2)
    return g[["correct", "rate"]].unstack("arm")

rate(runs)

correct           rate          
arm                                      both exec tool both exec tool
model                     task                                        
gemma4:12b                find-out        5/5  4/5  5/5  1.0  0.8  1.0
                          sales-total     3/5  5/5  0/5  0.6  1.0  0.0
gemma4:31b                find-out        5/5  5/5  5/5  1.0  1.0  1.0
                          sales-total     5/5  5/5  0/5  1.0  1.0  0.0
qwen3.8-flash-next:vision find-out        5/5  5/5  5/5  1.0  1.0  1.0
                          sales-total     5/5  4/5  0/5  1.0  0.8  0.0

Pooled over models, task x arm.

In [5]:
g = runs.groupby(["task", "arm"]).correct.agg(["sum", "count"])
g["rate"] = (g["sum"] / g["count"]).round(2)
g

sum  count  rate
task        arm                   
find-out    both   15     15  1.00
            exec   14     15  0.93
            tool   15     15  1.00
sales-total both   13     15  0.87
            exec   14     15  0.93
            tool    0     15  0.00

## Steps, tokens and run time: mean and median

In [6]:
runs["tokens"] = runs.prompt_tokens.fillna(0) + runs.completion_tokens.fillna(0)
runs.groupby(G)[["steps", "tokens", "run_s", "gen_s"]].agg(["mean", "median"]).round(1)

steps          tokens           run_s         gen_s       
                                            mean median     mean   median   mean median   mean median
model                     task        arm                                                            
gemma4:12b                find-out    both   2.2    2.0  16909.6  15879.0    7.7    7.8    7.2    7.0
                                      exec   5.6    6.0  30891.0  33252.0   20.6   20.3   19.6   18.9
                                      tool   2.4    3.0  12691.8  13854.0    7.2    7.0    6.6    6.8
                          sales-total both   3.2    3.0  23335.6  23889.0   32.6   25.1   23.6   24.8
                                      exec   2.0    2.0  12347.6  12135.0    8.3    7.0    7.6    6.6
                                      tool   3.8    4.0  18431.8  19962.0   95.6   44.0   33.9   28.6
gemma4:31b                find-out    both   1.0    1.0  10641.4  10648.0   10.0   10.3    9.5    9.7
                                      exec   4.0    4.0  24805.0  28002.0   30.1   31.2   29.4   29.5
                                      tool   1.0    1.0   7085.0   7089.0    9.6    9.9    9.2    9.4
                          sales-total both   4.6    5.0  37569.4  34338.0   53.2   54.3   52.3   53.4
                                      exec   5.2    5.0  30637.2  26284.0   37.8   34.9   36.3   33.2
                                      tool   9.8   10.0  54155.6  53260.0  242.7  223.6  220.4  216.9
qwen3.8-flash-next:vision find-out    both   3.6    3.0  28249.4  22348.0   11.8   10.2   10.6    8.6
                                      exec   5.4    5.0  30078.0  30266.0   28.6   28.2   27.2   27.3
                                      tool   3.6    3.0  18070.4  15477.0   16.9   15.4   16.4   14.8
                          sales-total both   3.4    4.0  24702.6  28093.0   14.3   13.8   13.5   12.9
                                      exec   5.4    5.0  30786.8  27964.0   31.7   34.6   30.3   32.5
                                      tool  13.2   15.0  35913.0  42383.0  300.3  300.3  238.4  273.2

Pooled over models, task x arm.

In [7]:
runs.groupby(["task", "arm"])[["steps", "tokens", "run_s", "gen_s"]].agg(["mean", "median"]).round(1)

steps          tokens           run_s         gen_s       
                  mean median     mean   median   mean median   mean median
task        arm                                                            
find-out    both   2.3    2.0  18600.1  15879.0    9.8    9.3    9.1    8.6
            exec   5.0    5.0  28591.3  30241.0   26.4   27.7   25.4   26.5
            tool   2.3    2.0  12615.7  10567.0   11.2   10.2   10.7    9.5
sales-total both   3.7    4.0  28535.9  27588.0   33.4   27.0   29.8   25.8
            exec   4.2    4.0  24590.5  21513.0   26.0   25.4   24.7   23.7
            tool   8.9   10.0  36166.8  42383.0  212.9  266.0  164.2  182.6

## Reader mix: the first URL reader of each run, all arms

In [8]:
runs.assign(first_reader=runs.first_reader.fillna("none")).pivot_table(
    index=G, columns="first_reader", values="repeat", aggfunc="count", fill_value=0)

first_reader                                fetch_url  ml.fetch  page-fetch
model                     task        arm                                  
gemma4:12b                find-out    both          5         0           0
                                      exec          0         5           0
                                      tool          5         0           0
                          sales-total both          5         0           0
                                      exec          0         1           4
                                      tool          5         0           0
gemma4:31b                find-out    both          5         0           0
                                      exec          0         5           0
                                      tool          5         0           0
                          sales-total both          4         1           0
                                      exec          0         5           0
                                      tool          5         0           0
qwen3.8-flash-next:vision find-out    both          5         0           0
                                      exec          0         5           0
                                      tool          5         0           0
                          sales-total both          5         0           0
                                      exec          0         5           0
                                      tool          5         0           0

## Exec arm: where ml.fetch appears

In [9]:
ex = runs[runs.arm == "exec"]
print("mlfetch_turn distribution (None = never used ml.fetch), by model:")
display(ex.assign(mlfetch_turn=ex.mlfetch_turn.fillna(-1).astype(int).replace(-1, "none")).pivot_table(
    index=["model", "task"], columns="mlfetch_turn", values="repeat", aggfunc="count", fill_value=0))
summary = ex.groupby(["model", "task"]).apply(lambda d: pd.Series({
    "n": len(d),
    "used_mlfetch": int(d.mlfetch_turn.notna().sum()),
    "docs_before_mlfetch": f"{int(d.docs_before_mlfetch.fillna(False).sum())}/{int(d.mlfetch_turn.notna().sum())}",
    "page_fetch_rate": round((d.first_reader == "page-fetch").mean(), 2),
    "any_page_fetch": int(d.readers.apply(lambda r: "page-fetch" in r).sum()),
    "failed_first_exec": int(d.failed_first_exec.fillna(False).sum()),
}), include_groups=False)
summary

mlfetch_turn distribution (None = never used ml.fetch), by model:


mlfetch_turn                           1  2  none
model                     task                   
gemma4:12b                find-out     3  2     0
                          sales-total  1  0     4
gemma4:31b                find-out     5  0     0
                          sales-total  5  0     0
qwen3.8-flash-next:vision find-out     5  0     0
                          sales-total  5  0     0

n  used_mlfetch docs_before_mlfetch  page_fetch_rate  any_page_fetch  failed_first_exec
model                     task                                                                                                
gemma4:12b                find-out     5             5                 0/5              0.0               0                  0
                          sales-total  5             1                 0/1              0.8               4                  0
gemma4:31b                find-out     5             5                 0/5              0.0               0                  1
                          sales-total  5             5                 0/5              0.0               1                  5
qwen3.8-flash-next:vision find-out     5             5                 0/5              0.0               0                  1
                          sales-total  5             5                 0/5              0.0               0                  2

Exec arm, pooled over models.

In [10]:
ex.groupby("task").apply(lambda d: pd.Series({
    "n": len(d), "used_mlfetch": int(d.mlfetch_turn.notna().sum()),
    "docs_before_mlfetch": int(d.docs_before_mlfetch.fillna(False).sum()),
    "page_fetch_first": int((d.first_reader == "page-fetch").sum()),
    "median_mlfetch_turn": d.mlfetch_turn.median(),
    "correct_when_mlfetch": f"{int(d[d.mlfetch_turn.notna()].correct.sum())}/{int(d.mlfetch_turn.notna().sum())}",
    "correct_when_page_fetch": f"{int(d[d.first_reader == 'page-fetch'].correct.sum())}/{int((d.first_reader == 'page-fetch').sum())}",
}), include_groups=False)

,n,used_mlfetch,docs_before_mlfetch,page_fetch_first,median_mlfetch_turn,correct_when_mlfetch,correct_when_page_fetch
task,,,,,,,
find-out,15,15,0,0,1.0,14/15,0/0
sales-total,15,11,0,4,1.0,10/11,4/4


What came before the first ml.fetch (the step kinds, exec arm).

In [11]:
ex[ex.mlfetch_turn.notna()][["model", "task", "repeat", "mlfetch_turn", "before_mlfetch", "correct"]].sort_values(["model", "task", "repeat"])

,model,task,repeat,mlfetch_turn,before_mlfetch,correct
5,gemma4:12b,find-out,0,1.0,[],True
6,gemma4:12b,find-out,1,1.0,[],True
7,gemma4:12b,find-out,2,2.0,[exec-other],True
8,gemma4:12b,find-out,3,1.0,[],False
9,gemma4:12b,find-out,4,2.0,[exec-other],True
54,gemma4:12b,sales-total,4,1.0,[],True
20,gemma4:31b,find-out,0,1.0,[],True
21,gemma4:31b,find-out,1,1.0,[],True
22,gemma4:31b,find-out,2,1.0,[],True
23,gemma4:31b,find-out,3,1.0,[],True


## Both arm: first reader and switching

In [12]:
bo = runs[runs.arm == "both"]
display(bo.assign(first_reader=bo.first_reader.fillna("none")).pivot_table(
    index=["model", "task"], columns="first_reader", values="repeat", aggfunc="count", fill_value=0))
bo.groupby(["model", "task"]).apply(lambda d: pd.Series({
    "n": len(d), "switched": int(d.switched.sum()), "switched_rate": round(d.switched.mean(), 2),
    "correct": f"{int(d.correct.sum())}/{len(d)}",
}), include_groups=False)

first_reader                           fetch_url  ml.fetch
model                     task                            
gemma4:12b                find-out             5         0
                          sales-total          5         0
gemma4:31b                find-out             5         0
                          sales-total          4         1
qwen3.8-flash-next:vision find-out             5         0
                          sales-total          5         0

n  switched  switched_rate correct
model                     task                                           
gemma4:12b                find-out     5         0            0.0     5/5
                          sales-total  5         4            0.8     3/5
gemma4:31b                find-out     5         0            0.0     5/5
                          sales-total  5         4            0.8     5/5
qwen3.8-flash-next:vision find-out     5         2            0.4     5/5
                          sales-total  5         5            1.0     5/5

Both arm, pooled over models: first reader against correctness.

In [13]:
bo.assign(first_reader=bo.first_reader.fillna("none")).groupby(["task", "first_reader"]).correct.agg(["sum", "count"])

sum  count
task        first_reader            
find-out    fetch_url      15     15
sales-total fetch_url      12     14
            ml.fetch        1      1

## Calls to a tool the arm did not offer

In [14]:
u = runs.groupby(G).unknown_tool_calls.agg(["sum", "count"]).rename(columns={"sum": "unknown_tool_calls", "count": "runs"})
display(u)
print("total:", int(runs.unknown_tool_calls.sum()), "in", int((runs.unknown_tool_calls > 0).sum()), "of", len(runs), "runs")

unknown_tool_calls  runs
model                     task        arm                           
gemma4:12b                find-out    both                   0     5
                                      exec                   0     5
                                      tool                   0     5
                          sales-total both                   0     5
                                      exec                   0     5
                                      tool                   4     5
gemma4:31b                find-out    both                   0     5
                                      exec                   0     5
                                      tool                   0     5
                          sales-total both                   0     5
                                      exec                   0     5
                                      tool                  11     5
qwen3.8-flash-next:vision find-out    both                   0     5
                                      exec                   0     5
                                      tool                   0     5
                          sales-total both                   0     5
                                      exec                   0     5
                                      tool                   0     5

total: 15 in 9 of 90 runs


## Runs that hit the 300 s cap

The runner ends a run at the spec's `timeoutMs` with no answer; it is scored as not correct above.

In [15]:
t = runs[runs.timed_out]
display(t.groupby(G).size().rename("timed out"))
print("total:", len(t), "of", len(runs), "runs; every one in:", sorted(set(zip(t.task, t.arm))))

model                      task         arm 
gemma4:12b                 sales-total  tool    1
gemma4:31b                 sales-total  tool    1
qwen3.8-flash-next:vision  sales-total  tool    5
Name: timed out, dtype: int64

total: 7 of 90 runs; every one in: [('sales-total', 'tool')]


## A relative URL to `ml.fetch`

The task says "/sales.csv on this site", and a model often passes exactly that. The worker parses the URL with no base
(`src/sw/sw-fetch-url.ts`), so a relative one is refused as "not a valid URL", even though the page has a base for it.

In [16]:
r = runs[runs.relative_mlfetch > 0]
print("relative ml.fetch calls:", int(runs.relative_mlfetch.sum()), "in", len(r), "runs; refused:", int(runs.relative_refused.sum()))
display(r[["model", "task", "arm", "repeat", "relative_mlfetch", "relative_refused", "correct"]].sort_values(["model", "task", "arm", "repeat"]))
print("of those runs, correct:", f"{int(r.correct.sum())}/{len(r)}")

relative ml.fetch calls: 9 in 9 runs; refused: 9


,model,task,arm,repeat,relative_mlfetch,relative_refused,correct
20,gemma4:31b,find-out,exec,0,1,1,True
65,gemma4:31b,sales-total,exec,0,1,1,True
66,gemma4:31b,sales-total,exec,1,1,1,True
67,gemma4:31b,sales-total,exec,2,1,1,True
68,gemma4:31b,sales-total,exec,3,1,1,True
69,gemma4:31b,sales-total,exec,4,1,1,True
38,qwen3.8-flash-next:vision,find-out,exec,3,1,1,True
83,qwen3.8-flash-next:vision,sales-total,exec,3,1,1,True
84,qwen3.8-flash-next:vision,sales-total,exec,4,1,1,True


of those runs, correct: 9/9
